In [28]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import urllib.request
import json
import warnings
from geopy.distance import geodesic
import sys
!{sys.executable} -m pip install folium
import folium
from IPython.display import display

warnings.filterwarnings('ignore')

# ==========================================
# STEP 1: โหลดข้อมูลที่คลีนแล้ว
# ==========================================
try:
    df_cleaned = pd.read_csv('../data/NY_House_Cleaned.csv')
    print("✅ 1. โหลดไฟล์สำเร็จ")
except:
          print("❌ ไม่พบไฟล์: ../data/NY_House_Cleaned.csv")


# ==========================================
# STEP 1.5: คำนวณระยะทาง (จำเป็นต้องใช้ใน Matchmaker)
# ==========================================
print("กำลังเตรียมข้อมูลพิกัดและระยะทาง...")
if 'DISTANCE_TO_CENTER' not in df_cleaned.columns:
    ECONOMIC_HUBS = [
        (40.7580, -73.9855), (40.7081, -74.0093), # Manhattan
        (40.6925, -73.9868),                      # Brooklyn
        (40.7447, -73.9485), (40.7654, -73.8282), (40.7024, -73.7966), # Queens
        (40.8162, -73.9165),                      # Bronx
        (40.6437, -74.0759)                       # Staten Island
    ]
    def min_distance_to_hubs(lat, lon):
        distances = [geodesic((lat, lon), (hub_lat, hub_lon)).miles for hub_lat, hub_lon in ECONOMIC_HUBS]
        return min(distances)

    df_cleaned['DISTANCE_TO_CENTER'] = df_cleaned.apply(lambda row: min_distance_to_hubs(row['LATITUDE'], row['LONGITUDE']), axis=1)

# ==========================================
# STEP 7: The Matchmaker (ระบบจับคู่บ้านด้วย AI)
# ==========================================
print("7. เปิดระบบ The Matchmaker กำลังค้นหาบ้านที่ใช่สำหรับคุณ...")

USER_MAX_PRICE = 850000
USER_MIN_BEDS = 2
USER_MIN_BATHS = 1
USER_IDEAL_SQFT = 1200
USER_IDEAL_DISTANCE = 3.0

print(f"💰 งบสูงสุด: ${USER_MAX_PRICE:,.0f} | 🛏️ ขั้นต่ำ: {USER_MIN_BEDS} นอน {USER_MIN_BATHS} น้ำ")

df_filtered = df_cleaned[
    (df_cleaned['PRICE'] <= USER_MAX_PRICE) &
    (df_cleaned['BEDS'] >= USER_MIN_BEDS) &
    (df_cleaned['BATH'] >= USER_MIN_BATHS)
].copy()

if len(df_filtered) == 0:
    print("ไม่พบบ้านที่ตรงกับเงื่อนไขพื้นฐานเลย ลองปรับงบหรือสเปคดูนะครับ")
else:

    features_to_match = ['PRICE', 'PROPERTYSQFT', 'DISTANCE_TO_CENTER']
    X_match = df_filtered[features_to_match].copy()

    scaler_match = StandardScaler()
    X_match_scaled = scaler_match.fit_transform(X_match)

    user_profile = pd.DataFrame([[USER_MAX_PRICE, USER_IDEAL_SQFT, USER_IDEAL_DISTANCE]], columns=features_to_match)
    user_profile_scaled = scaler_match.transform(user_profile)

    n_recommendations = min(5, len(df_filtered))
    knn = NearestNeighbors(n_neighbors=n_recommendations, metric='euclidean')
    knn.fit(X_match_scaled)

    distances, indices = knn.kneighbors(user_profile_scaled)

    top_matches = df_filtered.iloc[indices[0]].copy()
    top_matches['MATCH_SCORE'] = 100 - (distances[0] * 10)
    top_matches['MATCH_SCORE'] = top_matches['MATCH_SCORE'].clip(upper=99.99)

    print(f"\n✨ เจอแล้ว! นี่คือ Top {n_recommendations} บ้านที่ตรงสเปคคุณที่สุด:")
    display_cols = ['PRICE', 'BEDS', 'BATH', 'PROPERTYSQFT', 'DISTANCE_TO_CENTER', 'MATCH_SCORE']
    display(top_matches[display_cols].style.format({
        'PRICE': '${:,.0f}',
        'PROPERTYSQFT': '{:,.0f} sqft',
        'DISTANCE_TO_CENTER': '{:.2f} miles',
        'MATCH_SCORE': '{:.1f}% Match'
    }))

    map_center = [top_matches['LATITUDE'].mean(), top_matches['LONGITUDE'].mean()]
    m = folium.Map(location=map_center, zoom_start=12)

    for i, row in top_matches.iterrows():
        popup_info = f"<b>Price:</b> ${row['PRICE']:,.0f}<br><b>Area:</b> {row['PROPERTYSQFT']} sqft<br><b>Match:</b> {row['MATCH_SCORE']:.1f}%"
        folium.Marker(
            location=[row['LATITUDE'], row['LONGITUDE']],
            popup=popup_info,
            tooltip=f"Top Match! ({row['MATCH_SCORE']:.1f}%)",
            icon=folium.Icon(color='red', icon='heart')
        ).add_to(m)

    display(m)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ 1. โหลดไฟล์สำเร็จ
กำลังเตรียมข้อมูลพิกัดและระยะทาง...
7. เปิดระบบ The Matchmaker กำลังค้นหาบ้านที่ใช่สำหรับคุณ...
💰 งบสูงสุด: $850,000 | 🛏️ ขั้นต่ำ: 2 นอน 1 น้ำ

✨ เจอแล้ว! นี่คือ Top 5 บ้านที่ตรงสเปคคุณที่สุด:


,PRICE,BEDS,BATH,PROPERTYSQFT,DISTANCE_TO_CENTER,MATCH_SCORE
2016,"$839,000",3,2.000000,"1,120 sqft",2.61 miles,97.7% Match
1590,"$798,000",2,2.000000,"1,190 sqft",2.68 miles,97.0% Match
274,"$850,000",2,2.000000,"1,005 sqft",2.99 miles,96.9% Match
486,"$799,999",3,2.000000,"1,200 sqft",2.59 miles,96.8% Match
2433,"$850,000",3,2.000000,"1,090 sqft",3.65 miles,96.5% Match


In [29]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import urllib.request
import json
import warnings
from geopy.distance import geodesic
import folium
from IPython.display import display

warnings.filterwarnings('ignore')

# ==========================================
# STEP 1: โหลดข้อมูลที่คลีนแล้ว
# ==========================================
try:
    df_cleaned = pd.read_csv('../data/NY_House_Cleaned.csv')
    print("✅ 1. โหลดไฟล์สำเร็จ")
except:
          print("❌ ไม่พบไฟล์: ../data/NY_House_Cleaned.csv")


# ==========================================
# STEP 1.5: คำนวณระยะทาง (จำเป็นต้องใช้ใน Matchmaker)
# ==========================================
print("กำลังเตรียมข้อมูลพิกัดและระยะทาง...")

if 'DISTANCE_TO_CENTER' not in df_cleaned.columns:
    ECONOMIC_HUBS = [
        (40.7580, -73.9855), (40.7081, -74.0093), # Manhattan
        (40.6925, -73.9868),                      # Brooklyn
        (40.7447, -73.9485), (40.7654, -73.8282), (40.7024, -73.7966), # Queens
        (40.8162, -73.9165),                      # Bronx
        (40.6437, -74.0759)                       # Staten Island
    ]
    def min_distance_to_hubs(lat, lon):
        distances = [geodesic((lat, lon), (hub_lat, hub_lon)).miles for hub_lat, hub_lon in ECONOMIC_HUBS]
        return min(distances)

    df_cleaned['DISTANCE_TO_CENTER'] = df_cleaned.apply(lambda row: min_distance_to_hubs(row['LATITUDE'], row['LONGITUDE']), axis=1)

# ==========================================
# STEP 7: The Matchmaker (ระบบจับคู่บ้านด้วย AI)
# ==========================================
print("7. เปิดระบบ The Matchmaker กำลังค้นหาบ้านที่ใช่สำหรับคุณ...")

USER_MAX_PRICE = 850000
USER_MIN_BEDS = 2
USER_MIN_BATHS = 1
USER_IDEAL_SQFT = 1200
USER_IDEAL_DISTANCE = 3.0
USER_PREFERRED_CITY = 'Queens'

print(f"💰 งบ: ${USER_MAX_PRICE:,.0f} | 🛏️ ขั้นต่ำ: {USER_MIN_BEDS} นอน {USER_MIN_BATHS} น้ำ | 🏙️ โซน: {USER_PREFERRED_CITY}")

df_filtered = df_cleaned[
    (df_cleaned['PRICE'] <= USER_MAX_PRICE) &
    (df_cleaned['BEDS'] >= USER_MIN_BEDS) &
    (df_cleaned['BATH'] >= USER_MIN_BATHS)
].copy()

if USER_PREFERRED_CITY.lower() != 'any':

    possible_city_cols = [col for col in df_filtered.columns if col.lower() in ['locality', 'sublocality', 'borough', 'county', 'city']]

    if possible_city_cols:
        city_col = possible_city_cols[0]
        df_filtered = df_filtered[df_filtered[city_col].astype(str).str.contains(USER_PREFERRED_CITY, case=False, na=False)]
    else:
        address_cols = [col for col in df_filtered.columns if 'address' in col.lower()]
        if address_cols:
            addr_col = address_cols[0]
            df_filtered = df_filtered[df_filtered[addr_col].astype(str).str.contains(USER_PREFERRED_CITY, case=False, na=False)]
        else:
            print("ไม่พบคอลัมน์เขต/เมืองในข้อมูล ระบบจะข้ามการกรองโซนไป")

if len(df_filtered) == 0:
    print(f"ไม่พบบ้านที่ตรงกับเงื่อนไขในโซน {USER_PREFERRED_CITY} ลองปรับงบ สเปค หรือเปลี่ยนโซนดูนะครับ")
else:
    features_to_match = ['PRICE', 'PROPERTYSQFT', 'DISTANCE_TO_CENTER']
    X_match = df_filtered[features_to_match].copy()

    scaler_match = StandardScaler()
    X_match_scaled = scaler_match.fit_transform(X_match)

    user_profile = pd.DataFrame([[USER_MAX_PRICE, USER_IDEAL_SQFT, USER_IDEAL_DISTANCE]], columns=features_to_match)
    user_profile_scaled = scaler_match.transform(user_profile)

    n_recommendations = min(5, len(df_filtered))
    knn = NearestNeighbors(n_neighbors=n_recommendations, metric='euclidean')
    knn.fit(X_match_scaled)

    distances, indices = knn.kneighbors(user_profile_scaled)

    top_matches = df_filtered.iloc[indices[0]].copy()
    top_matches['MATCH_SCORE'] = 100 - (distances[0] * 10)
    top_matches['MATCH_SCORE'] = top_matches['MATCH_SCORE'].clip(upper=99.99)

    display_cols = ['PRICE', 'BEDS', 'BATH', 'PROPERTYSQFT', 'DISTANCE_TO_CENTER', 'MATCH_SCORE']
    possible_display_cols = [col for col in top_matches.columns if col.lower() in ['locality', 'sublocality', 'borough']]
    if possible_display_cols:
        display_cols.insert(1, possible_display_cols[0])

    print(f"\n✨ เจอแล้ว! นี่คือ Top {n_recommendations} บ้านที่ตรงสเปคคุณที่สุด (โซน {USER_PREFERRED_CITY}):")
    display(top_matches[display_cols].style.format({
        'PRICE': '${:,.0f}',
        'PROPERTYSQFT': '{:,.0f} sqft',
        'DISTANCE_TO_CENTER': '{:.2f} miles',
        'MATCH_SCORE': '{:.1f}% Match'
    }))

    map_center = [top_matches['LATITUDE'].mean(), top_matches['LONGITUDE'].mean()]
    m = folium.Map(location=map_center, zoom_start=12)

    for i, row in top_matches.iterrows():
        popup_info = f"<b>Price:</b> ${row['PRICE']:,.0f}<br><b>Area:</b> {row['PROPERTYSQFT']} sqft<br><b>Match:</b> {row['MATCH_SCORE']:.1f}%"
        folium.Marker(
            location=[row['LATITUDE'], row['LONGITUDE']],
            popup=popup_info,
            tooltip=f"Top Match! ({row['MATCH_SCORE']:.1f}%)",
            icon=folium.Icon(color='red', icon='heart')
        ).add_to(m)

    display(m)

✅ 1. โหลดไฟล์สำเร็จ
กำลังเตรียมข้อมูลพิกัดและระยะทาง...
7. เปิดระบบ The Matchmaker กำลังค้นหาบ้านที่ใช่สำหรับคุณ...
💰 งบ: $850,000 | 🛏️ ขั้นต่ำ: 2 นอน 1 น้ำ | 🏙️ โซน: Queens

✨ เจอแล้ว! นี่คือ Top 5 บ้านที่ตรงสเปคคุณที่สุด (โซน Queens):


,PRICE,LOCALITY,BEDS,BATH,PROPERTYSQFT,DISTANCE_TO_CENTER,MATCH_SCORE
1590,"$798,000",Queens County,2,2.000000,"1,190 sqft",2.68 miles,96.4% Match
486,"$799,999",Queens County,3,2.000000,"1,200 sqft",2.59 miles,96.1% Match
112,"$759,000",Queens County,3,2.000000,"1,200 sqft",2.64 miles,94.5% Match
3596,"$799,000",Queens County,2,2.000000,868 sqft,2.95 miles,94.1% Match
381,"$749,000",Queens County,2,2.000000,"1,300 sqft",2.70 miles,93.9% Match


In [30]:
# ==========================================
# STEP 8: Save Matchmaker Model
# ==========================================
import joblib
import os

print("💾 กำลังบันทึกโมเดล Matchmaker...")

model_dir = "../app/ml_models"
os.makedirs(model_dir, exist_ok=True)

# save model
joblib.dump(knn, f"{model_dir}/matchmaker_model.pkl")

# save scaler
joblib.dump(scaler_match, f"{model_dir}/matchmaker_scaler.pkl")

# save features
joblib.dump(features_to_match, f"{model_dir}/matchmaker_features.pkl")

print("✅ บันทึก Matchmaker Model สำเร็จ")


💾 กำลังบันทึกโมเดล Matchmaker...
✅ บันทึก Matchmaker Model สำเร็จ


In [36]:
import pandas as pd
import joblib
import folium
from IPython.display import display

# โหลด dataset
df = pd.read_csv("../data/NY_House_Cleaned.csv")

# โหลดโมเดล
model = joblib.load("../app/ml_models/matchmaker_model.pkl")
scaler = joblib.load("../app/ml_models/matchmaker_scaler.pkl")

features = ['PRICE', 'PROPERTYSQFT', 'DISTANCE_TO_CENTER']

# user input
user_input = pd.DataFrame([[850000,1200,3.0]], columns=features)

# scale
user_scaled = scaler.transform(user_input)

# หาเพื่อนบ้านที่ใกล้ที่สุด
distances, indices = model.kneighbors(user_scaled)

# บ้านที่แนะนำ
top_matches = df.iloc[indices[0]].copy()

# คำนวณ match score
top_matches['MATCH_SCORE'] = 100 - (distances[0] * 10)
top_matches['MATCH_SCORE'] = top_matches['MATCH_SCORE'].clip(upper=99.99)

# แสดงผล
print(top_matches[['PRICE','BEDS','BATH','PROPERTYSQFT','MATCH_SCORE']])

# ===============================
# แสดงแผนที่
# ===============================

map_center = [top_matches['LATITUDE'].mean(), top_matches['LONGITUDE'].mean()]

m = folium.Map(location=map_center, zoom_start=12)

for _, row in top_matches.iterrows():
    folium.Marker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        popup=f"Price: ${row['PRICE']:,.0f}",
        icon=folium.Icon(color="red")
    ).add_to(m)

# แสดงใน notebook
display(m)

# backup: เซฟเป็นไฟล์
m.save("../templates/map.html")


        PRICE  BEDS      BATH  PROPERTYSQFT  MATCH_SCORE
110  17500000     5  6.000000   5499.000000    96.418577
28     230000     1  1.000000   2184.207862    96.103033
4    55000000     7  2.373861  14175.000000    94.465470
251    875000     3  2.000000   1437.000000    94.105742
18     350000     1  1.000000    700.000000    93.900549
